# Exercise 4: Your First LangGraph

**Level:** Basic

LangGraph is a framework for building **stateful, multi-step AI workflows** as graphs. Unlike simple chains, graphs let you define exactly how data flows between steps, including loops and branches.

**What you will learn:**
- What LangGraph is and when to use it over simple chains
- Defining state with TypedDict
- Creating nodes (functions) and connecting them with edges
- Compiling and invoking a graph
- Visualizing the graph structure
## 1. Setup & Installation

In [ ]:
!pip install langgraph langchain langchain-google-genai -q
import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-key-here"

## 2. Why Graphs?

Simple chains are linear: `A → B → C`. But real agent workflows need:
- **Branching**: Take different paths based on conditions
- **Loops**: Retry or refine until a condition is met
- **State**: Track information across steps

LangGraph models these as a **directed graph** where:
- **Nodes** = functions that process state
- **Edges** = connections between nodes
- **State** = shared data that flows through the graph

```
START → [Node A] → [Node B] → [Node C] → END
                         ↓
                    [Node D] → END
```

## 3. Defining State

State is the data that flows through your graph. In LangGraph, you define it as a `TypedDict`.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# Simple state — just a dictionary with typed fields
class SimpleState(TypedDict):
    """State for our first graph."""
    input_text: str           # The original input
    processed_text: str       # After processing
    word_count: int           # Analysis result
    summary: str              # Final output

# LangGraph uses Annotated types with reducers for list fields
# The `add_messages` reducer appends new messages instead of replacing
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # Appends, doesn't replace
    context: str

print("State types defined!")
print(f"SimpleState fields: {list(SimpleState.__annotations__.keys())}")
print(f"ChatState fields: {list(ChatState.__annotations__.keys())}")

## 4. Building Your First Graph

Let's build a simple text processing pipeline as a graph.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Define state
class TextState(TypedDict):
    raw_text: str
    cleaned_text: str
    word_count: int
    analysis: str

# Define node functions — each takes state and returns partial state update
def clean_text(state: TextState) -> dict:
    """Node 1: Clean the input text."""
    raw = state["raw_text"]
    cleaned = raw.strip().lower()
    cleaned = " ".join(cleaned.split())  # Normalize whitespace
    print(f"  [clean_text] '{raw[:30]}...' → '{cleaned[:30]}...'")
    return {"cleaned_text": cleaned}

def count_words(state: TextState) -> dict:
    """Node 2: Count words in the cleaned text."""
    words = state["cleaned_text"].split()
    count = len(words)
    print(f"  [count_words] Found {count} words")
    return {"word_count": count}

def analyze_text(state: TextState) -> dict:
    """Node 3: Produce a simple analysis."""
    text = state["cleaned_text"]
    count = state["word_count"]
    
    if count < 10:
        length_desc = "short"
    elif count < 50:
        length_desc = "medium"
    else:
        length_desc = "long"
    
    unique_words = len(set(text.split()))
    analysis = f"Text is {length_desc} ({count} words, {unique_words} unique). Starts with: '{text[:40]}'"
    print(f"  [analyze_text] {analysis}")
    return {"analysis": analysis}

# Build the graph
builder = StateGraph(TextState)

# Add nodes
builder.add_node("clean", clean_text)
builder.add_node("count", count_words)
builder.add_node("analyze", analyze_text)

# Add edges (define the flow)
builder.add_edge(START, "clean")      # Start → clean
builder.add_edge("clean", "count")     # clean → count
builder.add_edge("count", "analyze")   # count → analyze
builder.add_edge("analyze", END)        # analyze → End

# Compile the graph
graph = builder.compile()

print("Graph compiled successfully!")

## 5. Invoking the Graph

In [ ]:
# Run the graph with input
print("Running graph...\n")

result = graph.invoke({
    "raw_text": "   The quick brown FOX   jumps   over the lazy DOG   "
})

print(f"\nFinal state:")
print(f"  raw_text: {result['raw_text']}")
print(f"  cleaned_text: {result['cleaned_text']}")
print(f"  word_count: {result['word_count']}")
print(f"  analysis: {result['analysis']}")

## 6. Visualizing the Graph

LangGraph can render your graph as a Mermaid diagram.

In [ ]:
# Get the Mermaid diagram representation
mermaid = graph.get_graph().draw_mermaid()
print("Mermaid diagram:")
print(mermaid)

In [ ]:
# Render it visually (works in Colab/Jupyter)
from IPython.display import display, Image

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"PNG rendering requires additional dependencies: {e}")
    print("You can paste the Mermaid code above into https://mermaid.live to visualize it.")

## 7. Graph with an LLM Node

Now let's build something more interesting — a graph that uses an LLM to process text.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class ArticleState(TypedDict):
    topic: str
    outline: str
    draft: str
    final_article: str

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
parser = StrOutputParser()

def generate_outline(state: ArticleState) -> dict:
    """Generate an article outline."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You create concise article outlines. Output only the outline, 3-4 bullet points."),
        ("human", "Create an outline for an article about: {topic}")
    ])
    chain = prompt | model | parser
    outline = chain.invoke({"topic": state["topic"]})
    print(f"  [outline] Generated outline")
    return {"outline": outline}

def write_draft(state: ArticleState) -> dict:
    """Write the article draft based on the outline."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a concise article writer. Write a short article (150 words max) following the given outline."),
        ("human", "Topic: {topic}\n\nOutline:\n{outline}\n\nWrite the article.")
    ])
    chain = prompt | model | parser
    draft = chain.invoke({"topic": state["topic"], "outline": state["outline"]})
    print(f"  [draft] Wrote draft ({len(draft.split())} words)")
    return {"draft": draft}

def polish_article(state: ArticleState) -> dict:
    """Polish and finalize the article."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an editor. Polish the article: fix grammar, improve flow, add a compelling title. Keep it concise."),
        ("human", "{draft}")
    ])
    chain = prompt | model | parser
    final = chain.invoke({"draft": state["draft"]})
    print(f"  [polish] Finalized article")
    return {"final_article": final}

# Build the graph
builder = StateGraph(ArticleState)
builder.add_node("outline", generate_outline)
builder.add_node("draft", write_draft)
builder.add_node("polish", polish_article)

builder.add_edge(START, "outline")
builder.add_edge("outline", "draft")
builder.add_edge("draft", "polish")
builder.add_edge("polish", END)

article_graph = builder.compile()
print("Article graph compiled!")

In [ ]:
# Run the article generation pipeline
print("Generating article...\n")
result = article_graph.invoke({"topic": "The future of AI agents in travel industry"})

print("\n" + "=" * 60)
print("OUTLINE:")
print(result["outline"])
print("\n" + "=" * 60)
print("FINAL ARTICLE:")
print(result["final_article"])

In [ ]:
# Visualize this graph
print(article_graph.get_graph().draw_mermaid())

## 8. Streaming Graph Execution

You can stream the graph's execution to see each node as it completes.

In [ ]:
# Stream node by node
print("Streaming execution...\n")

for event in article_graph.stream({"topic": "How AI is transforming airport operations"}):
    # event is a dict with the node name as key
    for node_name, node_output in event.items():
        print(f"\n--- Node: {node_name} ---")
        for key, value in node_output.items():
            preview = str(value)[:150] + "..." if len(str(value)) > 150 else str(value)
            print(f"  {key}: {preview}")

## 9. State Reducers — Accumulating Data

When a node returns a state update, by default it **replaces** the field. But sometimes you want to **append** (e.g., building a list of results). This is what `Annotated` reducers do.

In [ ]:
from typing import TypedDict, Annotated
import operator

# Custom state with a list reducer
class PipelineState(TypedDict):
    input: str
    steps_completed: Annotated[list[str], operator.add]  # Appends, doesn't replace
    result: str

def step_one(state: PipelineState) -> dict:
    return {
        "steps_completed": ["step_one"],
        "result": state["input"].upper()
    }

def step_two(state: PipelineState) -> dict:
    return {
        "steps_completed": ["step_two"],
        "result": state["result"] + "!!!"
    }

def step_three(state: PipelineState) -> dict:
    return {
        "steps_completed": ["step_three"],
        "result": f"[{state['result']}]"
    }

builder = StateGraph(PipelineState)
builder.add_node("one", step_one)
builder.add_node("two", step_two)
builder.add_node("three", step_three)

builder.add_edge(START, "one")
builder.add_edge("one", "two")
builder.add_edge("two", "three")
builder.add_edge("three", END)

pipeline = builder.compile()

result = pipeline.invoke({"input": "hello world", "steps_completed": []})
print(f"Result: {result['result']}")
print(f"Steps completed: {result['steps_completed']}")
# Notice steps_completed accumulated all three steps!

---
## YOUR TURN: Exercise A

Build a **data processing graph** with 4 nodes:
1. `fetch_data` — Returns a hardcoded list of flight prices: `[450, 320, 890, 200, 550, 310, 780]`
2. `filter_data` — Filters to flights under $500
3. `compute_stats` — Calculates min, max, average of filtered prices
4. `format_report` — Returns a formatted string report

Define appropriate state, connect the nodes, compile, invoke, and visualize.

In [ ]:
# YOUR TURN: Build the data processing graph

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# TODO: Define the state

# TODO: Define the 4 node functions

# TODO: Build, compile, and invoke the graph

# TODO: Print the Mermaid diagram

# TODO: Assert that the report contains correct values

---
## YOUR TURN: Exercise B

Build a **content pipeline graph** with LLM nodes:
1. `brainstorm` — Given a theme, generate 3 blog post ideas (use LLM)
2. `select_best` — Use the LLM to pick the best idea from the 3
3. `write_intro` — Use the LLM to write an intro paragraph for the selected idea

Use streaming to watch each node execute. Print the Mermaid diagram.

In [ ]:
# YOUR TURN: Build the content pipeline graph

# TODO: Define state with fields for theme, ideas, selected_idea, intro

# TODO: Define 3 LLM-powered node functions

# TODO: Build, compile, and stream the graph

# TODO: Print the Mermaid diagram

## Key Takeaways

- **LangGraph** models workflows as directed graphs with nodes and edges
- **State** (TypedDict) is shared data that flows through the graph
- **Nodes** are functions that take state and return partial updates
- **Edges** define the flow: `add_edge(source, target)`
- **Reducers** (via `Annotated`) control how state fields are updated (replace vs. append)
- Use `.stream()` to watch execution node by node
- Use `.get_graph().draw_mermaid()` to visualize the graph

**Next:** In Exercise 5, we will add conditional routing — branching and looping — to make our graphs truly dynamic.